# 3) Categorical & Numerical Feature Analysis

This notebook analyzes the categorical and numerical features of the healthcare disease prediction dataset before machine learning preprocessing.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import csv
from io import StringIO

In [ ]:
# Load dataset directly from GitHub
url = "https://raw.githubusercontent.com/vyasanbmathew2008/Team-6/main/healthcare-disease-prediction/dataset/healthcare_dataset.csv"

# Download CSV
response = requests.get(url)
response.raise_for_status()

# Read CSV without pandas' strict parser
reader = csv.reader(StringIO(response.text))
rows = list(reader)

# Separate header and data
header = rows[0]
data_rows = rows[1:]

# Find the maximum number of columns
max_fields = max(len(row) for row in data_rows)

# Add column names if extra fields exist
while len(header) < max_fields:
    header.append(f"Symptom_{len(header)}")

# Make all rows the same length
fixed_rows = []

for row in data_rows:
    if len(row) < max_fields:
        row = row + [""] * (max_fields - len(row))
    elif len(row) > max_fields:
        row = row[:max_fields]

    fixed_rows.append(row)

# Create DataFrame
df = pd.DataFrame(fixed_rows, columns=header)

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")

In [ ]:
# Display first five records
        
df.head()

## Categorical Feature Analysis

Categorical features contain labels, categories, names, or other textual information.

In this healthcare dataset, the disease and symptom fields are categorical features.

In [ ]:
# Identify categorical/text columns
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Categorical/Text Columns:\n")

for column in categorical_columns:
    print("-", column)

print(f"\nTotal categorical/text columns: {len(categorical_columns)}")

In [ ]:
# Display unique values and frequency counts for categorical columns

for column in categorical_columns:
    print(f"\n{column} - Unique Values: {df[column].nunique(dropna=True)}")
    print(df[column].value_counts(dropna=False).head(20))

In [ ]:
# Calculate cardinality of categorical features

categorical_summary = pd.DataFrame({
    "Column": categorical_columns,
    "Unique Values": [
        df[column].nunique(dropna=True)
        for column in categorical_columns
    ],
    "Unique Percentage": [
        (df[column].nunique(dropna=True) / len(df)) * 100
        for column in categorical_columns
    ]
})

categorical_summary

In [ ]:
# Identify low-cardinality categorical columns

low_cardinality_columns = [
    column
    for column in categorical_columns
    if df[column].nunique(dropna=True) <= 10
]

print("Low-Cardinality Categorical Columns:\n")

if low_cardinality_columns:
    for column in low_cardinality_columns:
        print("-", column)
else:
    print("No low-cardinality categorical columns found.")

print(
    f"\nTotal low-cardinality categorical columns: "
    f"{len(low_cardinality_columns)}"
)

In [ ]:
# Plot distributions of low-cardinality categorical features

for column in low_cardinality_columns:
    plt.figure(figsize=(8, 4))

    sns.countplot(
        data=df,
        x=column,
        order=df[column].value_counts().index
    )

    plt.title(f"Distribution of {column}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Disease Feature Analysis

The `Disease` column is the target variable of the healthcare disease prediction project.

The frequency of each disease is analyzed to understand the distribution of the target classes.

In [ ]:
# Analyze disease categories

disease_column = "Disease"

print("Number of unique diseases:", df[disease_column].nunique())
print("\nDisease categories:\n")
print(df[disease_column].unique())

In [ ]:
# Disease frequency table

disease_frequency = df[disease_column].value_counts().reset_index()
disease_frequency.columns = ["Disease", "Record_Count"]

disease_frequency

In [ ]:
# Plot disease distribution

plt.figure(figsize=(12, 8))

sns.countplot(
    data=df,
    y=disease_column,
    order=df[disease_column].value_counts().index
)

plt.title("Distribution of Diseases")
plt.xlabel("Number of Records")
plt.ylabel("Disease")
plt.tight_layout()
plt.show()

## Symptom Feature Analysis

The symptom columns contain categorical symptom names associated with each disease.

The number of unique symptoms and their frequencies are analyzed.

In [ ]:
# Identify symptom columns

symptom_columns = [
    column for column in df.columns
    if column.startswith("Symptom_")
]

print("Symptom Columns:\n")

for column in symptom_columns:
    print("-", column)

print(f"\nTotal symptom columns: {len(symptom_columns)}")

In [ ]:
# Count unique symptoms in each symptom column

symptom_summary = pd.DataFrame({
    "Column": symptom_columns,
    "Unique Symptoms": [
        df[column].replace("", np.nan).nunique(dropna=True)
        for column in symptom_columns
    ],
    "Missing Values": [
        (df[column] == "").sum()
        for column in symptom_columns
    ]
})

symptom_summary

In [ ]:
# Combine all symptom columns to calculate overall symptom frequency

all_symptoms = pd.concat(
    [df[column] for column in symptom_columns],
    ignore_index=True
)

all_symptoms = all_symptoms[all_symptoms != ""]

symptom_frequency = all_symptoms.value_counts().reset_index()
symptom_frequency.columns = ["Symptom", "Frequency"]

symptom_frequency.head(30)

In [ ]:
# Plot the most frequent symptoms

top_symptoms = symptom_frequency.head(20)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=top_symptoms,
    y="Symptom",
    x="Frequency"
)

plt.title("Top 20 Most Frequent Symptoms")
plt.xlabel("Frequency")
plt.ylabel("Symptom")
plt.tight_layout()
plt.show()

## Numerical Feature Analysis

Numerical features contain values represented by numbers.

The dataset is expected to contain mainly categorical disease and symptom features. Therefore, the numerical columns are checked before applying numerical analysis.

In [ ]:
# Identify numerical columns

numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numerical Columns:\n")

if numerical_columns:
    for column in numerical_columns:
        print("-", column)
else:
    print("No numerical columns found in the dataset.")

print(f"\nTotal numerical columns: {len(numerical_columns)}")

In [ ]:
# Generate descriptive statistics for numerical features

if numerical_columns:
    numerical_summary = df[numerical_columns].describe().T
    numerical_summary
else:
    print("No numerical features available for descriptive statistics.")

In [ ]:
# Check skewness of numerical features

if numerical_columns:
    skewness = (
        df[numerical_columns]
        .skew(numeric_only=True)
        .sort_values(ascending=False)
    )

    skewness_df = pd.DataFrame({
        "Column": skewness.index,
        "Skewness": skewness.values
    })

    skewness_df
else:
    print("No numerical features available for skewness analysis.")

In [ ]:
# Plot distributions of numerical features

if numerical_columns:
    for column in numerical_columns:
        plt.figure(figsize=(8, 4))

        sns.histplot(
            data=df,
            x=column,
            kde=True
        )

        plt.title(f"Distribution of {column}")
        plt.tight_layout()
        plt.show()
else:
    print("No numerical features available for distribution plots.")

## Categorical Encoding Consideration

Since the disease and symptom features are categorical, they cannot be directly supplied to most machine learning algorithms in their text form.

Possible preprocessing techniques include:

- Label Encoding for the target variable.
- Multi-label or binary encoding for symptoms.
- One-Hot Encoding where appropriate.

The actual encoding will be performed in a later preprocessing stage.

## Feature Analysis Summary

In [ ]:
# Generate feature analysis summary

print("=" * 55)
print("CATEGORICAL & NUMERICAL FEATURE ANALYSIS SUMMARY")
print("=" * 55)
print(f"Total records             : {len(df)}")
print(f"Total columns             : {len(df.columns)}")
print(f"Categorical/Text columns  : {len(categorical_columns)}")
print(f"Numerical columns         : {len(numerical_columns)}")
print(f"Disease classes           : {df[disease_column].nunique()}")
print(f"Symptom columns           : {len(symptom_columns)}")
print(f"Unique symptoms           : {all_symptoms.nunique()}")
print("=" * 55)

# Conclusion

The dataset was analyzed to identify categorical and numerical features. The Disease and Symptom columns were found to be categorical features suitable for categorical preprocessing and encoding.

The frequency distribution of diseases and symptoms was examined to understand the structure of the dataset. Numerical columns were also checked, and numerical analysis was performed only if numerical features were present.

These findings will support the next stages of data cleaning, feature engineering, encoding, and machine learning model development.